# Citation graph sanity check

Loads the latest snapshot for a field and looks at it hard enough to catch a
broken crawl before the model stage inherits the damage. Nothing here is
shipped code: if a check earns its keep it moves into `src/graph/build.py`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from graph.build import latest_snapshot, load_snapshot, summarize

FIELD = "gnn"
snapshot = latest_snapshot(ROOT / "data", FIELD)
graph = load_snapshot(snapshot)
print(snapshot.name, graph)

## Build provenance

What the cleaning stage threw away, and why.


In [ ]:
import json

provenance = json.loads((snapshot / "stats.json").read_text())
pd.Series(provenance["cleaning"]).to_frame("count")

## Shape of the graph

A citation graph should be a single weakly connected component, sparse, and
acyclic apart from the odd preprint-versus-publication artefact.


In [ ]:
stats = summarize(graph)
pd.Series({k: v for k, v in vars(stats).items() if k != "most_cited_within_graph"}).to_frame(
    "value"
)

## Degree distribution

Citation graphs are heavy-tailed. On log-log axes the in-degree distribution
should fall away roughly linearly; a bell shape means the crawl is truncated.


In [ ]:
in_degrees = pd.Series(dict(graph.in_degree()))
out_degrees = pd.Series(dict(graph.out_degree()))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, degrees, label in zip(axes, [in_degrees, out_degrees], ["in", "out"], strict=True):
    counts = degrees.value_counts().sort_index()
    ax.scatter(counts.index, counts.values, s=12)
    ax.set(xscale="log", yscale="log", xlabel=f"{label}-degree", ylabel="works")
    ax.set_title(f"{label}-degree distribution")
fig.tight_layout()

## Publication years

The seeds sit in the 2008-2017 band, so the mass of the graph should sit
*earlier* than that: the crawl walks backwards into what those papers cite.


In [ ]:
years = pd.Series(dict(graph.nodes(data="publication_year"))).dropna()
ax = years.astype(int).plot.hist(
    bins=range(int(years.min()), int(years.max()) + 2), figsize=(11, 3.5)
)
ax.set(xlabel="publication year", ylabel="works")

## Candidate originators

Raw in-degree is not the originator score - that arrives in PR 8 - but if the
most-cited works inside the graph are not recognisable ancestors of the field,
something is wrong upstream.


In [ ]:
pd.DataFrame(
    [
        {
            "id": node,
            "year": graph.nodes[node].get("publication_year"),
            "in_degree": degree,
            "title": graph.nodes[node].get("title"),
        }
        for node, degree in in_degrees.sort_values(ascending=False).head(20).items()
    ]
)

## Reachability from the seeds

Every node should be reachable from at least one seed, since that is how it
entered the crawl. A gap here means the cleaning stage cut too deep.


In [ ]:
import networkx as nx

from graph.config import load_field

seeds = [s for s in load_field(FIELD).seed_papers if s in graph]
reachable = set().union(*(nx.descendants(graph, seed) | {seed} for seed in seeds))
print(f"{len(seeds)} seeds survived cleaning")
print(f"{len(reachable)} / {graph.number_of_nodes()} nodes reachable from a seed")